In [0]:
# Configures the Bronze ingestion notebook, namespace, paths, timezone, and pipeline run identifier.

from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
VOLUME = "volume1"

NS = f"{CATALOG}.{SCHEMA}"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

SHIPMENTS_PATH = f"{BASE}/landing/shipments"
PRODUCTS_PATH = f"{BASE}/landing/product_limits"
DELIVERIES_PATH = f"{BASE}/landing/delivery_events"

SENSOR_LANDING = f"{BASE}/landing/sensor_events"
SENSOR_STAGING = f"{BASE}/staging/sensor_events"

SENSOR_CHECKPOINT = f"{BASE}/checkpoints/bronze_sensor_events"
SENSOR_SCHEMA_LOCATION = f"{BASE}/schemas/bronze_sensor_events"

BRONZE_PRODUCTS_TABLE = f"{NS}.bronze_product_limits"
BRONZE_SHIPMENTS_TABLE = f"{NS}.bronze_shipments"
BRONZE_DELIVERIES_TABLE = f"{NS}.bronze_delivery_events"
BRONZE_SENSOR_TABLE = f"{NS}.bronze_sensor_events"

RUN_ID = str(uuid.uuid4())

print("Namespace :", NS)
print("Base path :", BASE)
print("Run ID    :", RUN_ID)

In [0]:
# Defines explicit schemas for all Bronze source datasets.

product_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("min_temperature", DoubleType(), True),
    StructField("max_temperature", DoubleType(), True)
])

shipment_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True),
    StructField("dispatch_time", TimestampType(), True),
    StructField("expected_arrival", TimestampType(), True)
])

delivery_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("event_time", TimestampType(), True)
])

sensor_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("temperature_c", StringType(), True)
])

print("Bronze source schemas initialized")

In [0]:
products_raw = (
    spark.read
    .option("header", True)
    .schema(product_schema)
    .csv(PRODUCTS_PATH)
)

In [0]:
display(products_raw)

In [0]:
# Adds ingestion metadata to product records and writes the Bronze product Delta table.

products_bronze = (
    products_raw
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

(
    products_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PRODUCTS_TABLE)
)

print("Bronze product table written successfully")
print(
    "Bronze product rows:",
    spark.table(BRONZE_PRODUCTS_TABLE).count()
)

display(
    spark.table(BRONZE_PRODUCTS_TABLE)
)

In [0]:
display(products_bronze)

In [0]:
# Reads shipment source files from the landing directory using an explicit schema.

shipments_raw = (
    spark.read
    .option("header", True)
    .schema(shipment_schema)
    .csv(SHIPMENTS_PATH)
)

display(shipments_raw)

print("Shipment source rows:", shipments_raw.count())

In [0]:
# Adds ingestion metadata to shipment records and writes the Bronze shipment Delta table.

shipments_bronze = (
    shipments_raw
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

(
    shipments_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_SHIPMENTS_TABLE)
)

print("Bronze shipment table written successfully")
print(
    "Bronze shipment rows:",
    spark.table(BRONZE_SHIPMENTS_TABLE).count()
)

display(
    spark.table(BRONZE_SHIPMENTS_TABLE)
)

In [0]:
deliveries_raw = (
    spark.read
    .schema(delivery_schema)
    .json(DELIVERIES_PATH)
)

display(deliveries_raw)

In [0]:
# Adds ingestion metadata to delivery events and writes the Bronze delivery Delta table.

deliveries_bronze = (
    deliveries_raw
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

(
    deliveries_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DELIVERIES_TABLE)
)

print("Bronze delivery table written successfully")
print(
    "Bronze delivery rows:",
    spark.table(BRONZE_DELIVERIES_TABLE).count()
)

display(
    spark.table(BRONZE_DELIVERIES_TABLE)
)

In [0]:
SENSOR_LANDING = f"{BASE}/landing/sensor_events"
SENSOR_STAGING = f"{BASE}/staging/sensor_events"

SENSOR_CHECKPOINT = (
    f"{BASE}/checkpoints/bronze_sensor_events"
)

In [0]:
# Copies all available sensor JSON files from staging to landing without hardcoding batch names.

staging_files = [
    file_info
    for file_info in dbutils.fs.ls(SENSOR_STAGING)
    if file_info.name.endswith(".json")
]

landing_files = {
    file_info.name
    for file_info in dbutils.fs.ls(SENSOR_LANDING)
    if file_info.name.endswith(".json")
}

copied_files = 0
skipped_files = 0

for file_info in staging_files:
    if file_info.name not in landing_files:
        source_path = file_info.path
        destination_path = (
            f"{SENSOR_LANDING}/{file_info.name}"
        )

        dbutils.fs.cp(
            source_path,
            destination_path
        )

        copied_files += 1

        print(
            "Copied:",
            file_info.name
        )
    else:
        skipped_files += 1

        print(
            "Already in landing:",
            file_info.name
        )

print()
print("Files copied :", copied_files)
print("Files skipped:", skipped_files)

In [0]:
# Displays all JSON sensor files currently available in staging and landing.

staging_sensor_files = [
    file_info
    for file_info in dbutils.fs.ls(SENSOR_STAGING)
    if file_info.name.endswith(".json")
]

landing_sensor_files = [
    file_info
    for file_info in dbutils.fs.ls(SENSOR_LANDING)
    if file_info.name.endswith(".json")
]

print("STAGING FILES")
display(
    spark.createDataFrame(
        [
            (
                file_info.name,
                file_info.path,
                file_info.size
            )
            for file_info in staging_sensor_files
        ],
        [
            "file_name",
            "file_path",
            "size_bytes"
        ]
    )
)

print("LANDING FILES")
display(
    spark.createDataFrame(
        [
            (
                file_info.name,
                file_info.path,
                file_info.size
            )
            for file_info in landing_sensor_files
        ],
        [
            "file_name",
            "file_path",
            "size_bytes"
        ]
    )
)

print(
    "Sensor staging files:",
    len(staging_sensor_files)
)

print(
    "Sensor landing files:",
    len(landing_sensor_files)
)

In [0]:
%sql

SELECT *
FROM training_lab_classic.cold_chain.bronze_sensor_events;

In [0]:
# Creates a single Auto Loader stream for incremental sensor-file ingestion into Bronze.

sensor_stream = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        SENSOR_SCHEMA_LOCATION
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "rescue"
    )
    .option(
        "rescuedDataColumn",
        "_rescued_data"
    )
    .schema(sensor_schema)
    .load(SENSOR_LANDING)
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_file_name",
        F.col("_metadata.file_name")
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size")
    )
    .withColumn(
        "_source_file_modified_at",
        F.col("_metadata.file_modification_time")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

print("Sensor Auto Loader stream initialized")

In [0]:
# Clears old Bronze sensor rows, then writes the Auto Loader sensor stream into Bronze.
# availableNow processes all files in landing and stops; the checkpoint prevents reprocessing.

if spark.catalog.tableExists(BRONZE_SENSOR_TABLE) and not any(
    f.name.rstrip("/") == "commits" for f in (
        dbutils.fs.ls(SENSOR_CHECKPOINT) if any(
            p.name.rstrip("/") == "bronze_sensor_events"
            for p in dbutils.fs.ls(f"{BASE}/checkpoints")
        ) else []
    )
):
    spark.sql(f"DELETE FROM {BRONZE_SENSOR_TABLE}")
    print("Old Bronze sensor rows cleared (fresh checkpoint run).")

sensor_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SENSOR_SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(sensor_schema)
    .load(SENSOR_LANDING)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_file_name", F.col("_metadata.file_name"))
    .withColumn("_source_file_size", F.col("_metadata.file_size"))
    .withColumn("_source_file_modified_at", F.col("_metadata.file_modification_time"))
    .withColumn("_processing_time", F.current_timestamp())
    .withColumn("_run_id", F.lit(RUN_ID))
)

sensor_query = (
    sensor_stream.writeStream
    .format("delta")
    .option("checkpointLocation", SENSOR_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_SENSOR_TABLE)
)

sensor_query.awaitTermination()

print("Bronze sensor ingestion completed")
print("Bronze sensor rows:", spark.table(BRONZE_SENSOR_TABLE).count())

### verify that bad records survived

In [0]:
# Verifies the Bronze sensor table and reports the records ingested by the current pipeline run.

bronze_sensor = spark.table(
    BRONZE_SENSOR_TABLE
)

current_run_sensor = (
    bronze_sensor
    .filter(
        F.col("_run_id") == RUN_ID
    )
)

current_run_count = (
    current_run_sensor.count()
)

total_bronze_count = (
    bronze_sensor.count()
)

print("==============================================")
print("BRONZE SENSOR INGESTION SUMMARY")
print("==============================================")

print(
    "Run ID             :",
    RUN_ID
)

print(
    "Current run rows   :",
    current_run_count
)

print(
    "Total Bronze rows  :",
    total_bronze_count
)

display(
    current_run_sensor
    .orderBy(
        "_processing_time",
        "_source_file"
    )
)

In [0]:
# Profiles raw Bronze sensor data using only columns guaranteed by the sensor schema.

bronze_sensor = spark.table(
    BRONZE_SENSOR_TABLE
)

bronze_quality_profile = (
    bronze_sensor
    .select(
        F.count("*").alias(
            "total_rows"
        ),
        F.sum(
            F.when(
                F.col("event_id").isNull()
                | (F.trim(F.col("event_id")) == ""),
                1
            ).otherwise(0)
        ).alias(
            "missing_event_ids"
        ),
        F.sum(
            F.when(
                F.col("shipment_id").isNull()
                | (F.trim(F.col("shipment_id")) == ""),
                1
            ).otherwise(0)
        ).alias(
            "missing_shipment_ids"
        ),
        F.sum(
            F.when(
                F.col("sensor_id").isNull()
                | (F.trim(F.col("sensor_id")) == ""),
                1
            ).otherwise(0)
        ).alias(
            "missing_sensor_ids"
        ),
        F.sum(
            F.when(
                F.expr(
                    "try_cast(temperature_c AS DOUBLE)"
                ).isNull()
                & F.col("temperature_c").isNotNull(),
                1
            ).otherwise(0)
        ).alias(
            "nonnumeric_temperature_values"
        ),
        F.sum(
            F.when(
                F.expr(
                    "try_cast(event_time AS TIMESTAMP)"
                ).isNull()
                & F.col("event_time").isNotNull(),
                1
            ).otherwise(0)
        ).alias(
            "invalid_event_time_values"
        )
    )
)

display(bronze_quality_profile)

print("Bronze sensor profiling completed.")

In [0]:
# Displays the actual Bronze sensor schema and column names available to downstream processing.

bronze_sensor = spark.table(
    BRONZE_SENSOR_TABLE
)

bronze_sensor.printSchema()

print("Bronze sensor columns:")

for column_name in bronze_sensor.columns:
    print(column_name)

In [0]:
# Verifies that the Bronze sensor streaming checkpoint exists and contains ingestion state.

checkpoint_files = dbutils.fs.ls(
    SENSOR_CHECKPOINT
)

print(
    "Checkpoint location:",
    SENSOR_CHECKPOINT
)

print(
    "Checkpoint entries:",
    len(checkpoint_files)
)

display(
    checkpoint_files
)

### testing that rerun dosent add same entries

In [0]:
# Performs final structural checks for the Bronze tables created by this ingestion pipeline.

required_tables = [
    BRONZE_PRODUCTS_TABLE,
    BRONZE_SHIPMENTS_TABLE,
    BRONZE_DELIVERIES_TABLE,
    BRONZE_SENSOR_TABLE
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(
        table_name
    )
]

assert len(missing_tables) == 0, (
    "Missing Bronze tables: "
    + ", ".join(missing_tables)
)

product_bronze_count = (
    spark.table(
        BRONZE_PRODUCTS_TABLE
    ).count()
)

shipment_bronze_count = (
    spark.table(
        BRONZE_SHIPMENTS_TABLE
    ).count()
)

delivery_bronze_count = (
    spark.table(
        BRONZE_DELIVERIES_TABLE
    ).count()
)

sensor_bronze_count = (
    spark.table(
        BRONZE_SENSOR_TABLE
    ).count()
)

print("==============================================")
print("FINAL BRONZE READINESS REPORT")
print("==============================================")

print(
    "Products Bronze   :",
    product_bronze_count
)

print(
    "Shipments Bronze  :",
    shipment_bronze_count
)

print(
    "Deliveries Bronze :",
    delivery_bronze_count
)

print(
    "Sensors Bronze    :",
    sensor_bronze_count
)

print(
    "Required tables   :",
    len(required_tables)
)

print(
    "Missing tables    :",
    len(missing_tables)
)

print(
    "Run ID            :",
    RUN_ID
)

print()
print("Bronze ingestion readiness checks passed.")